<a href="https://colab.research.google.com/github/TienNguyen0712/hybrid-llm-tabular-pipeline-for-icu-mortality-prediction/blob/main/notebooks/predict_mortality_using_mimic_iv_baseline_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Xây dựng mô hình dự đoán tỷ lệ tử vong của ICU từ MIMIC-IV (BaseLine2)**

**Mục tiêu:**

Notebook này thực hiện xây dựng một mô hình dự đoán tỷ lệ tử vong từ bảng MIMIC bằng các mô hình học máy. Nhằm tạo ra một chuẩn dữ liệu để so sánh với các kỹ thuật khác trong tương lai (nếu có)

**Bộ dữ liệu**

MIMIC-IV là cơ sở dữ liệu lớn gồm các bảng về thông tin bệnh nhân, lâm sàng, các thủ thuật, v...

Lý do chọn bộ dữ liệu này do tính phức tạp, sát với dữ liệu thực tế, việc xử lý dữ liệu này là một trong những bước khó khắn, ...

- 2 module chính được sử dụng trong notebook này chính là `hosp` và `icu`. Chi tiết các bảng lựa chọn sẽ được mô tả ở dưới


## **1. Nạp thư viện và dữ liệu cần thiết**

In [1]:
# Bỏ comment nếu chạy trên Google Colab
from google.colab import drive
drive.mount('/content/drive')
!pip install tableone pyarrow seaborn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import OrderedDict
import gc
import warnings
warnings.filterwarnings('ignore')

from typing import List, Tuple, Dict, Optional

# Cài đặt style biểu đồ chuyên nghiệp
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
sns.set_style("whitegrid")
print("✓ Libraries loaded")


Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.0 MB/s eta 0:00:00
✓ Libraries loaded


In [ ]:
DATA_DIR = "/content/drive/MyDrive/NCKH-DDU1231/physionet.org/mimiciv/3.1"  # Google Colab dữ liệu chính

import os
def load(folder, name):
    path = os.path.join(DATA_DIR, folder, name)
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"  ✓ {folder}/{name}: {df.shape[0]:,} rows × {df.shape[1]} cols")
        return df
    else:
        print(f"  ✗ {folder}/{name}: KHÔNG TÌM THẤY")
        return pd.DataFrame()


In [2]:
DATA_DIR = "/content/drive/MyDrive/NCKH-DDU1231/mimic-iv-clinical-database-demo-2.2/mimic-iv-clinical-database-demo-2.2"  # Google Colab

import os
def load(folder, name):
    path = os.path.join(DATA_DIR, folder, name)
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"  ✓ {folder}/{name}: {df.shape[0]:,} rows × {df.shape[1]} cols")
        return df
    else:
        print(f"  ✗ {folder}/{name}: KHÔNG TÌM THẤY")
        return pd.DataFrame()

## **2. Tiền xử lý dữ liệu trước khi chia huấn luyện**

### **2.1. Xây dựng tập cohort**

Cohort được xây dựng bám sát theo các tiêu chí như sau:
- Phải là người trưởng thành (anchor_age > 18)
- Thời gian nằm ICU phải trên 24h (LOS >= 24)
- Chỉ lấy lần nhập ICU đầu tiên trong một lần nhập viện
- Loại các bệnh nhân đã tử vong, xuât hoặc chuyển viện trong vòng 24h đầu

In [4]:
def build_and_val_icu_cohort(patients_df: pd.DataFrame, admissions_df: pd.DataFrame, icustays_df: pd.DataFrame) -> pd.DataFrame:
  # Chuyển đổi định dạng mốc thời gian sang Datetime
  time_cols = {
      'icustays_df': ['intime', 'outtime'],
      'admissions_df': ['admittime', 'dischtime'],
      'patients_df' : []
  }

  for col in time_cols['icustays_df']:
        icustays_df[col] = pd.to_datetime(icustays_df[col])
  for col in time_cols['admissions_df']:
        admissions_df[col] = pd.to_datetime(admissions_df[col])
  # 1. Inner Join với admission
  cohort = icustays_df.merge(
  admissions_df[["subject_id", "hadm_id", "admittime", "dischtime",
                 "deathtime", "hospital_expire_flag"]],
          on=['subject_id', 'hadm_id'],
          how='inner'
      )
  # 2. Inner join với patients
  cohort = cohort.merge(
          patients_df[["subject_id", "gender", "anchor_age", "dod"]],
          on='subject_id',
          how='inner'
      )

  initial_count = len(cohort)
  print(f"Tổng số lượt ICU ban đầu: {initial_count:,}")

  # Áp dụng các tiêu chí lọc

  cohort = cohort[cohort["anchor_age"] >= 18].copy() # Bệnh nhân trên 18 tuổi
  cohort["Outcome"]   = cohort["hospital_expire_flag"].astype(int)
  cohort["Sex"]       = (cohort["gender"] == "M").astype(int)
  cohort["LOS_hours"] = cohort["los"].astype(float) * 24

  cohort["Outcome"]   = cohort["hospital_expire_flag"].astype(int)
  cohort["Sex"]       = (cohort["gender"] == "M").astype(int)
  cohort["LOS_hours"] = cohort["los"].astype(float) * 24
  cohort = cohort.rename(columns={
        "subject_id": "PatientID", "hadm_id": "AdmissionID",
        "stay_id": "StayID",       "intime":  "ICUInTime",
        "outtime": "ICUOutTime",   "anchor_age": "Age",
    })[["PatientID", "AdmissionID", "StayID",
        "ICUInTime", "ICUOutTime",
        "Age", "Sex", "Outcome", "LOS_hours"]].reset_index(drop=True)

  print(f"  ICU stays      : {len(cohort):,}")
  print(f"  Unique patients: {cohort['PatientID'].nunique():,}")
  print(f"  Mortality      : {cohort['Outcome'].sum():,}  "
        f"({cohort['Outcome'].mean()*100:.1f}%)")

  return cohort


In [5]:
print("Loading core tables...")
patients   = load("hosp", "patients.csv.gz")      # Thông tin bệnh nhân
admissions = load("hosp", "admissions.csv.gz")    # Thông tin nhập viện
icustays   = load("icu",  "icustays.csv.gz")      # Thông tin lần nằm

cohort = build_and_val_icu_cohort(patients, admissions, icustays)

Loading core tables...
  ✓ hosp/patients.csv.gz: 100 rows × 6 cols
  ✓ hosp/admissions.csv.gz: 275 rows × 16 cols
  ✓ icu/icustays.csv.gz: 140 rows × 8 cols
Tổng số lượt ICU ban đầu: 140
  ICU stays      : 140
  Unique patients: 100
  Mortality      : 20  (14.3%)


In [6]:
cohort.head()

,PatientID,AdmissionID,StayID,ICUInTime,ICUOutTime,Age,Sex,Outcome,LOS_hours
0,10018328,23786647,31269608,2154-04-24 23:03:44,2154-05-02 15:55:21,83,0,0,184.860278
1,10020187,24104168,37509585,2169-01-15 04:56:00,2169-01-20 15:47:50,63,0,0,130.863889
2,10020187,26842957,32554129,2170-02-24 18:18:46,2170-02-25 15:15:26,63,0,0,20.944444
3,10012853,27882036,31338022,2176-11-26 02:34:49,2176-11-29 20:58:54,91,0,0,90.401389
4,10020740,25826145,32145159,2150-06-03 20:12:32,2150-06-04 21:05:58,56,1,0,24.890556


In [7]:
cohort.isna().sum()

,0
PatientID,0
AdmissionID,0
StayID,0
ICUInTime,0
ICUOutTime,0
Age,0
Sex,0
Outcome,0
LOS_hours,0


In [8]:
cohort.columns

Index(['PatientID', 'AdmissionID', 'StayID', 'ICUInTime', 'ICUOutTime', 'Age',
       'Sex', 'Outcome', 'LOS_hours'],
      dtype='object')

## **3. Xây dựng các bảng đặc trưng thực hiện tiền xử lý cho dữ liệu huấn luyện**

In [ ]:
from typing import Dict, Tuple, List

### **3.1. Xây dựng bảng chart vitals từ `chartevent`**

Kết hợp với bảng cohort với cửa số được lấy trong vòng 24h khi nằm ở icu
- intime <= charttime <= intime + 24h
- Lọc các chỉ số theo mã id hợp lệ rồi học các giá trị ngoại lai
- Thực hiện sinh thêm các đặc trưng mới với các chỉ số phức tạp

In [ ]:
["subject_id", "hadm_id", "stay_id",
                     "charttime", "itemid", "value", "valuenum"],

In [10]:
# Mapping & Lists
CHART_ITEM2VAR = {
    220045: "HR", 220210: "RR", 224690: "RR", 224422: "RR",
    223761: "Temp_F", 223762: "Temp_C", 220179: "SBP", 220050: "SBP",
    220180: "DBP", 220051: "DBP", 220052: "MBP", 220181: "MBP", 220277: "SpO2",
    220739: "GCS_eye", 223901: "GCS_motor", 223900: "GCS_verbal",
    220546: "WBC", 220545: "HCT", 227457: "PLT", 220274: "pH", 223835: "FiO2",
    220224: "PaO2", 220235: "PaCO2", 227443: "HCO3", 227445: "CKMB", 227446: "BNP",
    220587: "AST", 220644: "ALT", 225690: "Bili_total", 225651: "Bili_direct",
    225636: "DDimer", 225624: "BUN", 220615: "Creatinine_raw", 227463: "Cortisol",
    220645: "Na", 227442: "K", 220602: "Cl", 220635: "Mg", 220621: "Glucose",
    226707: "Height_in", 226730: "Height_cm", 226512: "Weight_kg", 226531: "Weight_lbs",
}

GCS_EYE_MAP = {"None": 0, "1 No Response": 1, "2 To pain": 2, "To Pain": 2, "3 To speech": 3, "To Speech": 3, "4 Spontaneously": 4, "Spontaneously": 4}
GCS_MOTOR_MAP = {"1 No Response": 1, "No response": 1, "2 Abnorm extensn": 2, "Abnormal extension": 2, "3 Abnorm flexion": 3, "Abnormal Flexion": 3, "4 Flex-withdraws": 4, "Flex-withdraws": 4, "5 Localizes Pain": 5, "Localizes Pain": 5, "6 Obeys Commands": 6, "Obeys Commands": 6}
GCS_VERBAL_MAP = {"No Response-ETT": 1, "No Response": 1, "1 No Response": 1, "1.0 ET/Trach": 1, "2 Incomp sounds": 2, "Incomprehensible sounds": 2, "3 Inapprop words": 3, "Inappropriate Words": 3, "4 Confused": 4, "Confused": 4, "5 Oriented": 5, "Oriented": 5}

LAB_ITEM2VAR = {
    51301: "WBC_lab", 51221: "HCT_lab", 51265: "PLT_lab", 50820: "pH_lab", 50821: "PaO2_lab", 50818: "PaCO2_lab",
    50882: "HCO3_lab", 50911: "CKMB_lab", 50963: "BNP_lab", 50878: "AST_lab", 50861: "ALT_lab", 50862: "Albumin_lab",
    50885: "Bili_total_lab", 50883: "Bili_direct_lab", 50884: "Bili_indirect_lab", 51274: "PT_lab", 51275: "PTT_lab",
    50915: "DDimer_lab", 51006: "BUN_lab", 50912: "Creatinine_lab", 50983: "Na_lab", 50971: "K_lab", 50902: "Cl_lab",
    50893: "Ca_lab", 50960: "Mg_lab", 50993: "TSH_lab", 51001: "FT3_lab", 50995: "FT4_lab", 50889: "CRP_lab", 50813: "Lactate_lab",
}

VASOPRESSOR_ITEMS = {221289: "Epinephrine", 221906: "Norepinephrine", 221653: "Dobutamine", 221662: "Dopamine"}
URINE_ITEMS = [226559, 226560, 226561, 226563, 226564, 226565, 226567, 226631, 226632, 227489]
CRRT_ITEM = 225802

CHART_FEATURES = ["HR", "RR", "Temp_C", "SBP", "DBP", "MBP", "SpO2", "GCS_total", "WBC", "HCT", "PLT", "pH", "FiO2", "PaO2", "PaCO2", "HCO3", "CKMB", "BNP", "AST", "ALT", "Bili_total", "Bili_direct", "DDimer", "BUN", "Creatinine", "Cortisol", "Na", "K", "Cl", "Mg", "Glucose", "Height", "Weight"]
LAB_FEATURES = ["WBC_lab", "HCT_lab", "PLT_lab", "pH_lab", "PaO2_lab", "PaCO2_lab", "HCO3_lab", "CKMB_lab", "BNP_lab", "AST_lab", "ALT_lab", "Albumin_lab", "Bili_total_lab", "Bili_direct_lab", "Bili_indirect_lab", "PT_lab", "PTT_lab", "DDimer_lab", "BUN_lab", "Creatinine_lab", "Na_lab", "K_lab", "Cl_lab", "Ca_lab", "Mg_lab", "TSH_lab", "FT3_lab", "FT4_lab", "CRP_lab", "Lactate_lab"]
TREATMENT_FEATURES = ["Epinephrine_used", "Norepinephrine_used", "Dobutamine_used", "Dopamine_used", "Urine_output_mL", "CRRT_active"]
ALL_FEATURES = CHART_FEATURES + LAB_FEATURES + TREATMENT_FEATURES

VALID_RANGE = {
    "HR": (10, 300), "RR": (2, 80), "Temp_C": (25, 45), "Temp_F": (77, 113), "SBP": (30, 300), "DBP": (10, 200), "MBP": (20, 250), "SpO2": (50, 100),
    "GCS_total": (3, 15), "WBC": (0.1, 500), "HCT": (5, 70), "PLT": (1, 3000), "pH": (6.5, 8.5), "FiO2": (0.1, 1.05), "PaO2": (10, 700), "PaCO2": (10, 200),
    "HCO3": (5, 60), "CKMB": (0, 10000), "BNP": (1, 50000), "AST": (1, 50000), "ALT": (1, 50000), "Bili_total": (0.1, 100), "Bili_direct": (0.1, 100),
    "DDimer": (0.1, 100000), "BUN": (1, 500), "Creatinine": (0.1, 50), "Cortisol": (0.1, 200), "Na": (100, 180), "K": (1, 10), "Cl": (70, 150),
    "Mg": (0.5, 10), "Glucose": (10, 2000), "Height": (50, 250), "Weight": (10, 400), "WBC_lab": (0.1, 500), "HCT_lab": (5, 70), "PLT_lab": (1, 3000),
    "pH_lab": (6.5, 8.5), "PaO2_lab": (10, 700), "PaCO2_lab": (10, 200), "HCO3_lab": (5, 60), "Creatinine_lab": (0.1, 50), "BUN_lab": (1, 500),
    "Albumin_lab": (0.5, 8), "Lactate_lab": (0.1, 30), "CRP_lab": (0, 500), "Ca_lab": (3, 20), "Na_lab": (100, 180), "K_lab": (1, 10), "Cl_lab": (70, 150), "Mg_lab": (0.5, 10),
}

print(f"Tổng số features: {len(ALL_FEATURES)}")

Tổng số features: 69


In [12]:
def process_chartevents(df: pd.DataFrame, cohort: pd.DataFrame) -> pd.DataFrame:
  print("STEP 2: Xử lý chartevents...")
  stay2intime = cohort.set_index("StayID")["ICUInTime"].to_dict()
  valid_stays = set(cohort["StayID"].tolist())
  valid_items = set(CHART_ITEM2VAR.keys())